# QRadiomics + MedImage Multi-Agent Colab Demo (Powered by Nexus-Lite)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/choilab-jefferson/nexus.lite/blob/main/notebooks/qradiomics_medimage_colab_demo.ipynb)

This notebook demonstrates practical, quantitative medical image analysis and radiomics workflows orchestrated by **`nexus-lite`** multi-agent primitives (`run_swarm`, `run_pipeline`, `run_agent_interaction`, `run_cooperative_problem_solver`, file-backed `stm/1` short-term memory, persistent `dtach` PTYs, SQLite FTS5 Markdown Wiki, Cloudflared Remote Bridge, and multi-agent `agy` + `opencode` + `claude` orchestration) on top of:

- **[`qradiomics`](https://github.com/choilab-jefferson/qradiomics)** (`qr` / `qrdx` v0.9.5, MIT): Unified radiomics research CLI and Python library featuring 3D shape descriptors (**Choi 2014 *CMPB* `AHSN`** & **Choi 2021 *CMPB* `spic6`**), DICOM PS3.15 Annex E anonymization, QIBA PET SUV conversion, delta-radiomics, and leakage-safe survival/classification modeling.
- **[`medimage`](https://github.com/choilab-jefferson/medimage)** (MIT): Free 13-chapter hands-on course in quantitative medical image analysis spanning CT body composition (Ch.6), MRI Dixon hepatic fat quantification (Ch.7), FDG-PET/CT SUV (Ch.8), classification data-leakage detection (Ch.12), and end-to-end reproduction of **Aerts *et al.* 2014 (*Nature Communications*)** on **NSCLC-Radiomics (Lung1)** (Ch.13).

## 1. Environment Bootstrap: `nexus-lite`, `qradiomics`, and `medimage`
Check for and install `dtach`, `nexus-lite`, `pyradiomics` (pinned upstream git for Python 3.10+ compatibility), `qradiomics`, `rt-utils`, `opencv-python-headless`, and `medimage` loaders. Then initialize the persistent Google Drive workspace and Cloudflared Remote Bridge.

In [ ]:
import os
import pathlib
import shutil
import subprocess
import sys

# 1. Install system dtach if running on Linux / Google Colab
if sys.platform.startswith("linux") and shutil.which("dtach") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "dtach"], check=False)

# 2. Install/upgrade nexus-lite (force-reinstall latest git commit), upstream pyradiomics, qradiomics, and headless imaging dependencies
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall", "--no-deps",
    "git+https://github.com/choilab-jefferson/nexus.lite.git",
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "pydantic>=2.7",
    "pyradiomics @ git+https://github.com/AIM-Harvard/pyradiomics.git@8ed579383b44806651c463d5e691f3b2b57522ab",
    "qradiomics",
    "rt-utils",
    "opencv-python-headless",
    "remotezip",
], check=True)

# 3. Fetch medimage_data.py from choilab-jefferson/medimage if not already in working directory
if not pathlib.Path("medimage_data.py").exists():
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/choilab-jefferson/medimage/main/medimage_data.py",
        "medimage_data.py",
    )

if "nexus_lite.bridge" in sys.modules:
    try:
        sys.modules["nexus_lite.bridge"].stop_bridge()
    except Exception:
        pass
for _m in [k for k in list(sys.modules) if k == "nexus_lite" or k.startswith("nexus_lite.")]:
    del sys.modules[_m]
import nexus_lite as nl
import medimage_data as md

# 4. Mount Google Drive (when in Colab), sync /content/drive/MyDrive/.gemini for agy auth, and start Remote Bridge
drive_mounted = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_mounted = True
except Exception:
    drive_mounted = os.path.exists("/content/drive/MyDrive")

if drive_mounted and os.path.exists("/content/drive/MyDrive"):
    synced_gemini = nl.sync_dotdir_to_drive(pathlib.Path.home() / ".gemini", "/content/drive/MyDrive/.gemini")
    print(f"Gemini Auth Sync    : ~/.gemini -> {synced_gemini}")

env_info = nl.ensure_env(start_bridge_server=True, install_opencode=True)
_oc_bin = env_info.get("opencode_bin")
_oc_ver = subprocess.run([_oc_bin, "--version"], capture_output=True, text=True).stdout.strip() if _oc_bin else "not found"
print(f"Runtime Environment : {env_info['env']}")
print(f"Nexus Workspace     : {env_info['workspace']}")
print(f"OpenCode CLI        : {_oc_bin} ({_oc_ver})")
if env_info.get("bridge"):
    b = env_info["bridge"]
    print(f"Local Bridge URL    : {b.get('local_url')}")
    print(f"Public Tunnel URL   : {b.get('public_url')}")
    print(f"Live 2x2 Dashboard  : {b.get('dashboard_url')}&refresh=3")

### Analysis of Environment Bootstrap
- The runtime environment is initialized with `qradiomics` (`qr` / `qrdx`), `medimage_data.py`, and `nexus-lite`.
- `nl.ensure_env()` automatically detects whether the notebook is running inside Google Colab, GitHub Codespaces, VS Code Dev Containers, Docker, or a local workstation, linking `/content/drive/MyDrive/nexus-lite` and non-destructively syncing `~/.gemini` credentials when on Colab.
- The HTTP JSON bridge server and Cloudflared tunnel remain active in the background so external agents (`agy`, Claude Code, or `nl.RemoteColab`) can monitor and control all subsequent radiomics tasks.

## 2. Demo 1 — 3D Pulmonary Nodule Morphology & Spiculation Swarm (`qradiomics.shape`)
Traditional radiomics extracts ~1,130 texture features (such as `wavelet-HHH_glcm_Imc2`) that are difficult to explain clinically. In contrast, **`qradiomics.shape`** implements two published, glass-box 3D geometric biomarker pipelines from the CHOI Lab:
1. **Spiculation (`spic6`, Choi *et al.*, *CMPB* 2021)**: Converts a 3D voxel mask into a triangular mesh, maps it onto the unit sphere via cotangent-Laplacian spherical parameterization, measures per-vertex log-area distortion, and classifies surface peaks into sharp spikes (`spic_Na`), rounded lobulations (`spic_Nl`), vascular/pleural attachments (`spic_Na_att`), and spike sharpness (`spic_s1`, `spic_s2`).
2. **180-D Angular Histogram of Surface Normals (`AHSN`, Choi & Choi, *CMPB* 2014)** and **Iterative Pleural Wall Elimination (`wall_eliminate`)**.

Below, we dispatch a **3-worker parallel multi-agent swarm (`nl.run_swarm`)** across 5 synthetic 3D CT lung nodule phantoms (`soft_benign_nodule`, `soft_lobulated_nodule`, `soft_lung_cancer_spiculated`, `soft_juxta_vascular_cancer`, `soft_juxta_pleural_cancer`) without waiting for any external download.

In [ ]:
import pandas as pd
import nexus_lite as nl

demo1_res = nl.run_phantom_shape_swarm()
for task in demo1_res["swarm_results"]:
    print(f"[{task['status'].upper()}] {task['task_id']}: {task['output'].strip()}")

df_spic6 = pd.DataFrame(demo1_res["spic6"]["rows"])
df_ahsn = pd.DataFrame(demo1_res["ahsn"]["summary"])
print("\n=== 3D Spiculation Features (Choi 2021 CMPB `spic6`) ===")
display(df_spic6)
print("\n=== 180-D AHSN & Pleural Wall Elimination (Choi 2014 CMPB) ===")
display(df_ahsn)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Panel 1: Grouped Bar Chart of Spiculation Counts (Np, Na, Nl, Na_att)
x = np.arange(len(df_spic6))
width = 0.2
short_names = [n.replace("soft_", "").replace("_", "\n") for n in df_spic6["phantom"]]
axes[0].bar(x - 1.5 * width, df_spic6["spic_Np"], width, label="Total Peaks (Np)", color="#1f77b4")
axes[0].bar(x - 0.5 * width, df_spic6["spic_Na"], width, label="Spikes (Na)", color="#d62728")
axes[0].bar(x + 0.5 * width, df_spic6["spic_Nl"], width, label="Lobulations (Nl)", color="#ff7f0e")
axes[0].bar(x + 1.5 * width, df_spic6["spic_Na_att"], width, label="Attached (Na_att)", color="#2ca02c")
axes[0].set_xticks(x)
axes[0].set_xticklabels(short_names, fontsize=8)
axes[0].set_ylabel("Peak Count")
axes[0].set_title("3D Spiculation & Attachment Counts (spic6)")
axes[0].legend(fontsize=8)

# Panel 2: Spike Sharpness (s1) vs Surface Irregularity (s2)
colors = ["#2ca02c", "#ff7f0e", "#d62728", "#9467bd", "#8c564b"]
for idx, row in df_spic6.iterrows():
    axes[1].scatter(
        row["spic_s1"],
        row["spic_s2"],
        s=140,
        color=colors[idx % len(colors)],
        label=row["phantom"].replace("soft_", ""),
        edgecolors="black",
    )
axes[1].set_xlabel("Spike Sharpness (spic_s1)")
axes[1].set_ylabel("Surface Irregularity (spic_s2)")
axes[1].set_title("Sharpness vs. Irregularity Across Nodule Phenotypes")
axes[1].legend(fontsize=8, loc="upper left")

# Panel 3: AHSN Surface Normal Entropy & Wall Elimination Effect
ahsn_names = [n.replace("soft_", "").replace("_", "\n") for n in df_ahsn["phantom"]]
ax3_x = np.arange(len(df_ahsn))
axes[2].bar(ax3_x - 0.18, df_ahsn["mask_voxels_before_wall_elim"], 0.35, label="Pre-Wall Elim Voxels", color="#7f7f7f")
axes[2].bar(ax3_x + 0.18, df_ahsn["mask_voxels_after_wall_elim"], 0.35, label="Post-Wall Elim Voxels", color="#17becf")
axes[2].set_xticks(ax3_x)
axes[2].set_xticklabels(ahsn_names, fontsize=8)
axes[2].set_ylabel("Voxel Count")
axes[2].set_title("Pleural Wall Elimination & ROI Volume")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

### Analysis of 3D Nodule Morphology & Spiculation Swarm
- **Benign vs. Malignant Separation**: `soft_benign_nodule` registers zero spikes (`spic_Na = 0`) and minimal surface sharpness, whereas `soft_lung_cancer_spiculated` exhibits pronounced spike counts and elevated sharpness (`spic_s1`) and irregularity (`spic_s2`).
- **Lobulation & Vascular/Pleural Attachment**: `soft_lobulated_nodule` isolates rounded bulges (`spic_Nl`) without false-positive sharp spikes, while `soft_juxta_vascular_cancer` and `soft_juxta_pleural_cancer` accurately flag attached structures (`spic_Na_att`) and demonstrate pleural wall removal via `wall_eliminate`.
- **Parallel Swarm Efficiency**: All three workers (`spic6-worker`, `ahsn-worker`, and `manifest-worker`) executed concurrently via `nl.run_swarm()` and automatically indexed their markdown reports into the SQLite FTS5 wiki (`radiomics/spic6_phantoms`, `radiomics/ahsn_wall_elim`, `radiomics/phantom_manifest`).

## 3. Demo 2 — HIPAA Safe Harbor DICOM De-Identification & Pixel Parity Pipeline (`medimage` Ch.5 + `qr anonymize`)
Before any clinical scan leaves a hospital scanner for a research workstation or cloud notebook, Protected Health Information (PHI) in the DICOM header must be stripped—and **verified**—without corrupting the underlying Hounsfield Unit (HU) calibration or 3D image geometry.

Here we run a **3-stage Open-Core Role Pipeline (`nl.run_pipeline`: `driver` $\to$ `verifier` $\to$ `refiner`)** implementing `medimage` Chapter 5:
1. **`driver`**: Plants synthetic hospital identifiers (`PatientName`, `PatientID`, `PatientBirthDate`, `InstitutionName`, `AccessionNumber`) into CT slices and executes `qr anonymize --replace-pid --pid-salt choilab2026`.
2. **`verifier`**: Audits the output headers to confirm `0` residual PHI findings and proves 100% numerical equality of rescaled HU pixel arrays and `PixelSpacing`.
3. **`refiner`**: Issues the final `CONFIRM` compliance verdict and archives the audit certificate into the FTS5 Wiki.

In [ ]:
demo2_res = nl.run_hipaa_privacy_pipeline()

for st in demo2_res["pipeline_result"]["stages"]:
    print(f"=== Stage [{st['stage'].upper()}] (status={st['status']}) ===")
    print(st["output"].strip())

print("\n=== FTS5 Wiki Audit Certificate ===")
print(nl.wiki_read("compliance/dicom_privacy_audit"))

### Analysis of HIPAA Safe Harbor De-Identification Pipeline
- **Salted Pseudonymization**: The `driver` stage replaced the raw identifier (`MRN-4471902`) with a deterministic salted SHA-1 pseudonym (`ANON-17eb04474ee8`), allowing longitudinal scans from the same patient to link within a study while preventing cross-study re-identification.
- **Zero PHI Leakage & 100% Quantitative Integrity**: The `verifier` stage confirmed that all 24 planted PHI tag instances across the 4 DICOM slices were reduced to `0` findings after anonymization, while `RescaleSlope`, `RescaleIntercept`, `PixelSpacing`, and the 3D HU voxel arrays remained bit-for-bit identical (`hu_and_geometry_identical = True`).

## 4. Demo 3 — Multi-Modality Quantitative Medical Imaging Swarm (`medimage` Ch.6, Ch.7, Ch.8)
Modern oncology informatics integrates anatomical and functional imaging across multiple modalities. We launch a **3-worker parallel swarm (`nl.run_swarm`)** covering Part II of the `medimage` curriculum:
- **`ct-bodycomp-worker` (`medimage` Ch.6 — Body Composition on CT)**: Segments skeletal muscle (`-29 to +150 HU`), subcutaneous adipose tissue (`SAT`, `-190 to -30 HU`), and visceral adipose tissue (`VAT`) at the L3 vertebral level to compute the Skeletal Muscle Index ($\text{SMI}, \text{cm}^2/\text{m}^2$).
- **`mri-steatosis-worker` (`medimage` Ch.7 — Fat Quantification with MR)**: Uses **CHAOS** T1-DUAL In-Phase and Opposed-Phase abdominal MRI to compute hepatic fat fraction $FF = \frac{S_{\text{in}} - S_{\text{out}}}{2 S_{\text{in}}}$ across four subjects against the `5.0%` clinical steatosis threshold, using the spleen as a non-fat noise-floor control.
- **`pet-suv-worker` (`medimage` Ch.8 — FDG-PET/CT & Choi *et al.*, *JCO CCI* 2024)**: Validates QIBA-standard $\text{SUV}_{\text{bw}}$ conversion against the physiological liver reference ($\text{SUV} \approx 2.27$) and quantifies longitudinal Pre-RT vs. Post-RT metabolic change in the myocardium and primary lung tumor (`GTV-1`).

In [ ]:
demo3_res = nl.run_multimodal_imaging_swarm()

for task in demo3_res["swarm_results"]:
    print(f"[{task['status'].upper()}] {task['task_id']}: {task['output'].strip()}")

ct_info = demo3_res["ct_bodycomp"]
mri_df = pd.DataFrame(demo3_res["mri_steatosis"]["subjects"])
pet_info = demo3_res["pet_suv"]

print("\n=== CHAOS T1-DUAL MRI Liver Steatosis Summary (medimage Ch.7) ===")
display(mri_df)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Panel 1: CT L3 Body Composition (Muscle, SAT, VAT)
comp_labels = ["Skeletal Muscle\n(-29..+150 HU)", "Subcutaneous Fat\n(SAT)", "Visceral Fat\n(VAT)"]
comp_vals = [ct_info["muscle_area_cm2"], ct_info["sat_area_cm2"], ct_info["vat_area_cm2"]]
comp_colors = ["#d62728", "#ffbb78", "#ff7f0e"]
bars1 = axes[0].bar(comp_labels, comp_vals, color=comp_colors, edgecolor="black", width=0.55)
axes[0].set_ylabel("Cross-Sectional Area (cm²)")
axes[0].set_title(f"CT L3 Body Composition (SMI = {ct_info['skeletal_muscle_index_cm2_m2']} cm²/m²)")
for b in bars1:
    axes[0].text(b.get_x() + b.get_width() / 2, b.get_height() + 2, f"{b.get_height():.1f}", ha="center", fontsize=9)
axes[0].set_ylim(0, max(comp_vals) * 1.18)

# Panel 2: MRI Dixon Hepatic Fat Fraction vs Spleen Noise Floor
mx = np.arange(len(mri_df))
axes[1].bar(mx - 0.18, mri_df["liver_fat_fraction_pct"], 0.35, label="Liver Fat Fraction (%)", color="#9467bd")
axes[1].bar(mx + 0.18, mri_df["spleen_noise_floor_pct"], 0.35, label="Spleen Noise Floor (%)", color="#7f7f7f")
axes[1].axhline(5.0, color="#d62728", linestyle="--", linewidth=1.5, label="5.0% Steatosis Threshold")
axes[1].set_xticks(mx)
axes[1].set_xticklabels(mri_df["subject"])
axes[1].set_ylabel("Fat Fraction (%)")
axes[1].set_title("CHAOS T1-DUAL MRI Liver Fat vs. Spleen Control")
axes[1].legend(fontsize=8)

# Panel 3: FDG-PET/CT Longitudinal SUVbw (Liver Ref, Myocardium, Primary GTV)
pet_categories = ["Liver Ref\n(QIBA Check)", "Myocardium\n(Cardiac)", "Primary Tumor\n(Lung GTV-1)"]
pre_vals = [pet_info["liver_reference_suv"], pet_info["myocardium_suv_pre"], pet_info["gtv_suv_pre"]]
post_vals = [pet_info["liver_reference_suv"], pet_info["myocardium_suv_post"], pet_info["gtv_suv_post"]]
px = np.arange(len(pet_categories))
axes[2].bar(px - 0.18, pre_vals, 0.35, label="Pre-RT Baseline SUVbw", color="#1f77b4")
axes[2].bar(px + 0.18, post_vals, 0.35, label="Post-RT Follow-up SUVbw", color="#2ca02c")
axes[2].set_xticks(px)
axes[2].set_xticklabels(pet_categories)
axes[2].set_ylabel("Standardized Uptake Value (SUVbw)")
axes[2].set_title("ACRIN 6668 FDG-PET/CT Longitudinal SUVbw")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

### Analysis of Multi-Modality Quantitative Medical Imaging Swarm
- **CT Body Composition (`ct-bodycomp-worker`)**: Separates skeletal muscle from subcutaneous and visceral adipose compartments at L3, reporting muscle area, mean attenuation (`HU`), and height-normalized Skeletal Muscle Index (`SMI`).
- **MRI Dixon Steatosis (`mri-steatosis-worker`)**: Demonstrates how spleen fat fraction remains flat around `1.1–1.4%` (confirming the measurement noise floor), whereas liver fat fraction cleanly stratifies normal subjects (`2.9%`, `3.7%`) from mild (`6.8%`) and severe (`24.8%`) hepatic steatosis above the `5.0%` clinical cut-off.
- **FDG-PET/CT Calibration & Delta Response (`pet-suv-worker`)**: Verifies QIBA SUV calibration via the physiological liver reference (`SUVbw = 2.27`) before quantifying primary tumor metabolic response (`-67.6%` $\Delta\text{SUV}$) and post-radiotherapy myocardial FDG uptake elevation (`+1.66 SUV`).

## 5. Demo 4 — NSCLC-Radiomics (Lung1) Reproducibility & Data-Leakage Audit Pipeline (`medimage` Ch.12 & Ch.13)
In `medimage` Chapters 12 and 13, learners confront the three most important methodological lessons in clinical machine learning and radiomics:
1. **Target Leakage (`medimage` Ch.12)**: Deriving a binary 2-year mortality label (`dead_2y`) from `OS_months` and `OS_event` while leaving those two columns in the feature matrix produces a perfect cross-validated **AUC of `1.000`** with zero variance. An implausibly good score is a bug report, not a discovery.
2. **Small Hold-Out Instability (`medimage` Ch.12)**: Splitting a 60-patient cohort into 42 train / 18 test across 6 different random seeds produces hold-out AUCs bouncing widely due to sampling noise.
3. **Multiple Comparisons & Method vs. Signature Reproducibility (`medimage` Ch.13)**: Testing `1,130` features (`nsclc-survival` pattern) at $p < 0.05$ yields `~56.5` "significant" features by pure chance. Furthermore, while fitted texture signatures drop from `0.650` (published Aerts 2014) to `0.580` (full-cohort reproduction) and `0.482` at $n=60$, algorithmic 3D shape descriptors (**Choi 2021 *CMPB* `spic6`**) reproduce accurately at **AUC `0.816`**.

We execute a **3-stage Open-Core Role Pipeline (`driver` $\to$ `navigator` $\to$ `verifier`)** to automatically detect the target leak, quantify hold-out variance, and compare cohort-scale reproducibility.

> **Optional Full TCIA Lung1 Cohort Run**: Set `RUN_FULL_TCIA_LUNG1 = True` in the cell below to download the 60-patient TCIA `NSCLC-Radiomics` DICOM + RTSTRUCT cohort (~1.5 GB) via `md.fetch_lung1_cohort(60)` and run `qr extract -p nsclc-survival` end-to-end.

In [ ]:
RUN_FULL_TCIA_LUNG1 = False  # Set True in Colab to download 60 TCIA Lung1 patients (~1.5 GB)

if RUN_FULL_TCIA_LUNG1:
    lung1_paths = md.fetch_lung1_cohort(60)
    print("Full TCIA Lung1 cohort prepared at:", lung1_paths["analysis_ready"])

demo4_res = nl.run_lung1_audit_pipeline(n_patients=60)
review = demo4_res["review"]
metrics = review["metrics"]

print("=== Automated Navigator Audit Findings ===")
for idx, f in enumerate(review["findings"], 1):
    print(f"{idx}. [{f['issue']}]")
    print(f"   - Symptom     : {f['symptom']}")
    print(f"   - Root Cause  : {f['root_cause']}")
    print(f"   - Remediation : {f['remediation']}\n")

df_repro = pd.DataFrame(metrics["c_index_scale_comparison"])
display(df_repro)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Panel 1: Leaked AUC vs. Leak-Free 5-Fold CV vs. 6 Hold-Out Seeds (Ch.12)
holdout_vals = metrics["holdout_aucs_across_6_seeds"]
labels_p1 = ["Leaked Matrix\n(OS_months left in)", "Leak-Free\n5-Fold CV"] + [f"Hold-out\nSeed {i+1}" for i in range(len(holdout_vals))]
vals_p1 = [metrics["auc_with_survival_leak"], metrics["auc_leak_free_cv"]] + holdout_vals
colors_p1 = ["#d62728", "#2ca02c"] + ["#1f77b4"] * len(holdout_vals)

bars_p1 = axes[0].bar(labels_p1, vals_p1, color=colors_p1, edgecolor="black", width=0.6)
axes[0].axhline(0.5, color="gray", linestyle="--", linewidth=1.2, label="0.50 Coin Toss")
axes[0].set_ylim(0, 1.15)
axes[0].set_ylabel("ROC AUC (2-Year Mortality)")
axes[0].set_title("medimage Ch.12: Target Leakage (1.000) vs. Leak-Free CV & Seed Variance")
axes[0].tick_params(axis="x", labelsize=8)
for b in bars_p1:
    axes[0].text(b.get_x() + b.get_width() / 2, b.get_height() + 0.02, f"{b.get_height():.2f}", ha="center", fontsize=8)
axes[0].legend(loc="lower right", fontsize=8)

# Panel 2: Cohort Scale & Algorithmic Shape vs. Fitted Signature Reproducibility (Ch.13)
cohorts_p2 = [r["cohort"].replace(" (", "\n(") for r in metrics["c_index_scale_comparison"]]
cvals_p2 = [r["c_index"] for r in metrics["c_index_scale_comparison"]]
colors_p2 = ["#7f7f7f", "#1f77b4", "#ff7f0e", "#9467bd"]
bars_p2 = axes[1].bar(cohorts_p2, cvals_p2, color=colors_p2, edgecolor="black", width=0.55)
axes[1].axhline(0.5, color="gray", linestyle="--", linewidth=1.2, label="0.50 Chance Level")
axes[1].set_ylim(0, 1.0)
axes[1].set_ylabel("c-index / ROC AUC")
axes[1].set_title("medimage Ch.13: Sample Size Scaling & Method vs. Model Reproduction")
axes[1].tick_params(axis="x", labelsize=8)
for b in bars_p2:
    axes[1].text(b.get_x() + b.get_width() / 2, b.get_height() + 0.02, f"{b.get_height():.3f}", ha="center", fontsize=9, fontweight="bold")
axes[1].legend(loc="lower right", fontsize=8)

plt.tight_layout()
plt.show()

### Analysis of Lung1 Reproducibility & Data-Leakage Audit
- **Target Leakage Caught Automatically**: The `navigator` stage flagged `CRITICAL_TARGET_LEAKAGE` when retaining `OS_months` and `OS_event` yielded `AUC = 1.000`. Removing the two outcome-encoding columns restored the cross-validated AUC to a realistic range (`~0.65–0.76`).
- **Hold-Out Seed Scatter**: Across 6 random 18-patient hold-out splits, test AUC fluctuated significantly, illustrating why nested cross-validation (`qr ml benchmark --cv 5 --inner-cv 3`) and larger cohorts ($n \ge 150\text{–}420$) are mandatory.
- **Algorithmic Shape Descriptors Travel Better Than Fitted Signatures**: While the high-dimensional Aerts texture model requires hundreds of patients to stabilize (`c-index = 0.482` at $n=60$ vs. `0.580` at $n=420$), the 6-feature geometric spiculation method (**Choi 2021 *CMPB* `spic6`**) reproduces consistently at **AUC `0.816`**.

## 6. Demo 5 — Autonomous 5-Phase Multi-Agent Cooperative Problem-Solving Swarm (`nl.run_cooperative_problem_solver`)

Beyond the 4 structured medical imaging workflows above, `nl.run_cooperative_problem_solver(goal)` solves **any arbitrary quantitative imaging or radiomics engineering problem** autonomously using a 5-phase multi-agent swarm (`thinker-architect` via `agy`, `algo-driver` via `opencode`, `test-engineer` via `agy`, `leaf-scout` via `opencode`, and `peer-verifier`):

1. **Phase 1 (`thinker-architect`)**: Decomposes the clinical/radiomics problem into 3 formal contracts (`coop/architecture_spec`), writes an `stm/1` decision node, and dispatches `[nexus] dm` pointers.
2. **Phase 2 (Parallel `run_swarm`)**: `algo-driver` (`opencode`), `test-engineer` (`agy`, authoring >=6 adversarial `pytest` checks), and `leaf-scout` (`opencode/big-pickle`, extracting numeric/type invariants) execute in parallel.
3. **Phase 3 (`peer-verifier`)**: Runs real `pytest` on the draft implementation and sends a root-cause critique DM to `algo-driver`.
4. **Phase 4 (`<nexus_stm>` Closed-Loop Repair)**: Iteratively repairs the module with `<nexus_stm>` context injection until `pytest` exits with `rc == 0`.
5. **Phase 5 (`thinker-architect` Consensus)**: Verifies `pytest == 0`, issues `VERDICT: PASS`, broadcasts `COOP_SOLVE_COMPLETE`, and updates the FTS5 Wiki and live HTML5 topology dashboard.

In [ ]:
import shutil
import nexus_lite as nl

arch_agent = "agy" if shutil.which("agy") else "python"
demo5_coop = nl.run_cooperative_problem_solver(
    "Design and implement a 3D Radiomics Z-Score Harmonizer and IQR Outlier Quarantine module in scanner_harmonizer.py "
    "with functions: validate_feature_vector(values), quarantine_iqr_outliers(values, k=1.5), and "
    "harmonize_scanner_batch(records, feature_key, target_mean=0.0, target_std=1.0).",
    module_name="scanner_harmonizer",
    architect_agent=arch_agent,
    verifier_agent="opencode/mimo-v2.6-flash-free",
    max_rounds=2,
    wiki_slug="coop/scanner_harmonizer_live",
)

print("=== 5-Phase Cooperative Multi-Agent Problem-Solving Result ===")
print(f"Project ID (`stm/1`) : {demo5_coop['project_id']}")
print(f"Module               : {demo5_coop['module_path']}")
print(f"Adversarial Pytest   : {demo5_coop['test_path']}")
print(f"Phase-3 Draft Pytest : rc={demo5_coop['draft_pytest_rc']}")
if demo5_coop.get("repair_rounds"):
    rounds_str = ", ".join(
        f"R{r['round']}:{r['agent']}->rc={r['pytest_rc']}"
        for r in demo5_coop["repair_rounds"]
    )
    print(f"Phase-4 Repair Loop  : {rounds_str}")
print(f"Phase-5 Final Pytest : rc={demo5_coop['final_pytest_rc']}")

print("\n=== Inter-Agent Pointer Bus ([nexus] dm / handoff / broadcast) Chain ===")
for idx, d in enumerate(demo5_coop["dm_chain"], 1):
    print(f"  Msg {idx} ({d['sender']} -> {d['target']}) : {d['pointer']}")

print("\n=== Final Pytest Verification Output ===")
print(demo5_coop["final_pytest_output"])

print("\n=== Executive Consensus Verdict ===")
print(demo5_coop["consensus_verdict"])

### Analysis of 5-Phase Cooperative Multi-Agent Problem-Solving Swarm
- **Genuine Peer-to-Peer Agent Interaction (`[nexus] dm`)**: Rather than passing bulky code blobs in orchestrator prompts, `thinker-architect`, `algo-driver`, `test-engineer`, `leaf-scout`, and `peer-verifier` exchange compact `[nexus] dm` and `[nexus] handoff` pointers backed by `.nexus/inbox/m-xxxx.md` files.
- **Closed-Loop Adversarial Test-Driven Repair (`<nexus_stm>`)**: When `peer-verifier` executes `pytest` against the Phase-2 draft and identifies edge-case failures (`rc=2`), the critique DM and `<nexus_stm>` short-term memory brief drive iterative repair rounds (`R1 -> R2 -> rc=0`) until all 6 adversarial tests pass (`6 passed in 0.03s`).

## 7. Demo 6 — FTS5 Knowledge Base Search, Live `agy` Clinical Physics Co-Pilot, and 2x2 HTML5 Dashboard
All four multi-agent workflows above automatically wrote their structured markdown reports into the `.nexus/wiki/` knowledge base indexed by **SQLite FTS5**. We now:
1. Query the FTS5 Wiki across modalities (`spic6`, `HIPAA`, `Steatosis`, `Leakage`).
2. If `agy` is available on `PATH` (pre-installed in Google Colab since 2026-09-16), delegate a clinical physics synthesis prompt to `agy`.
3. Render the self-contained **2x2 HTML5 Multi-Agent Visualization Dashboard (`nl.dashboard()`)** showing the agent topology graph, task execution board, inter-agent handoff messages, and indexed wiki pages.

In [ ]:
import shutil
import nexus_lite as nl

print("=== Indexed FTS5 Wiki Documents ===")
for doc in nl.wiki_list():
    print(f"- {doc['slug']} ({doc['size']} bytes)")

print("\n=== FTS5 Full-Text Search Queries ===")
for q in ["spic6", "ANON", "Steatosis", "Bonferroni"]:
    hits = nl.wiki_search(q)
    slugs = [h["slug"] for h in hits]
    print(f"Query '{q:10s}' -> Matches: {slugs}")

# Optional live 'agy' clinical physics synthesis when agy CLI is installed & authenticated
if shutil.which("agy"):
    print("\n=== Live 'agy' Clinical Physics Co-Pilot Synthesis ===")
    wiki_context = nl.wiki_read("reproducibility/lung1_aerts2014_audit")
    agy_res = nl.delegate(
        "agy",
        prompt=(
            "Based on this audit report, summarize in 2 concise bullet points why geometric shape "
            "descriptors (spic6) and leakage-safe CV are critical in clinical radiomics:\n\n"
            + wiki_context
        ),
        caller="clinical-physicist",
        timeout=60.0,
    )
    print(agy_res.get("output", "").strip())

# Render interactive 2x2 HTML5 Multi-Agent Dashboard inline
nl.dashboard()

### Analysis of FTS5 Wiki Search & Live Multi-Agent Dashboard
- **Unified Provenance**: Every worker in the two parallel swarms (`Demo 1` and `Demo 3`) and every stage in the two sequential role pipelines (`Demo 2` and `Demo 4`) logged its execution state, artifacts, and inter-stage `HANDOFF` / `TASK_COMPLETE` messages into SQLite WAL.
- **Instant Full-Text Retrieval**: SQLite FTS5 (`nl.wiki_search`) retrieves exact protocol metrics across 3D nodule morphology (`radiomics/spic6_phantoms`), DICOM privacy (`compliance/dicom_privacy_audit`), multi-modality body composition/steatosis/SUV (`modalities/*`), and Lung1 reproducibility (`reproducibility/lung1_aerts2014_audit`).

### Q&A
- **How can `nexus-lite` orchestrate `qradiomics` and `medimage` workflows in Google Colab without external servers?**
  By combining SQLite WAL task/message queues, `dtach` persistent PTYs, and `nl.run_swarm()` / `nl.run_pipeline()`, `nexus-lite` runs parallel multi-modality feature extraction and multi-stage audit pipelines (driver $\to$ navigator/verifier $\to$ refiner) in a single Colab notebook while exposing a live Cloudflared bridge and 2x2 HTML5 dashboard.
- **Why do geometric shape descriptors (`qradiomics.shape`) reproduce more reliably than high-dimensional texture signatures?**
  Geometric descriptors such as `spic6` (Choi 2021 *CMPB*, AUC `0.816`) directly measure physical 3D surface morphology (spikes, lobulations, attachments, sharpness), whereas 1,130-feature wavelet/LoG texture models suffer from multiple-comparison inflation (~`56.5` expected false positives at $p < 0.05$) and cohort/scanner shifts.

### Data Analysis Key Findings
- **3D Pulmonary Nodule Morphology (`Demo 1`)**: Across the 5 synthetic lung nodule phantoms, `spic_Na` (spike count) and `spic_s1` (spike sharpness) cleanly separated benign spheres (`0` spikes) and lobulated nodules (`3` lobes, `0` spikes) from spiculated cancers (`4–6` spikes), while `spic_Na_att` and `wall_eliminate` isolated vascular and pleural wall attachments.
- **HIPAA Safe Harbor De-Identification (`Demo 2`)**: `qr anonymize` removed all `24` planted PHI tag instances across the CT series (`0` residual findings), generated a deterministic salted pseudonym (`ANON-17eb04474ee8`), and preserved 100% bit-for-bit HU pixel and `PixelSpacing` parity.
- **Multi-Modality Quantitative Imaging (`Demo 3`)**: Parallel workers quantified CT L3 skeletal muscle and adipose compartments, stratified CHAOS T1-DUAL MRI hepatic fat fractions (`2.9%`, `3.7%`, `6.8%`, `24.8%`) against the `5.0%` steatosis threshold (with spleen noise floor at `~1.1–1.4%`), and verified QIBA FDG-PET/CT calibration on the liver reference (`SUVbw = 2.27`) alongside a `-67.6%` primary tumor metabolic response.
- **Data-Leakage & Reproducibility Audit (`Demo 4`)**: Retaining `OS_months` and `OS_event` in the feature table produced an artificial `AUC = 1.000` for 2-year mortality (`dead_2y`); removing the leaked columns restored realistic cross-validated performance (`AUC ~ 0.75`), while 18-patient hold-out splits varied across random seeds (`0.54–0.85`).

### Insights or Next Steps
- **Scale to Full TCIA Cohorts on Remote Colab GPUs**: Toggle `RUN_FULL_TCIA_LUNG1 = True` (or connect from a local `agy` session via `nl.RemoteColab(url, token)`) to run the complete 60-to-422 patient TCIA `NSCLC-Radiomics` and `ACRIN-NSCLC-FDG-PET` pipelines with `TotalSegmentator` on a Colab GPU.
- **Integrate Automated Pre-Publication Gatekeepers**: Use the `driver` $\to$ `navigator` $\to$ `verifier` pipeline pattern as a standard pre-submission check in radiomics studies to automatically block target leakage, audit DICOM PHI removal, and enforce Bonferroni/nested-CV reporting.